# CKIP要的東西

In [ ]:
!pip install ckiptagger
!pip install ckipnlp
!pip install ckip_classic

In [ ]:
import os
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from ckipnlp.pipeline import CkipPipeline, CkipDocument

pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER

In [ ]:
ws = WS("/content/drive/MyDrive/CKIP/data")
pos = POS("/content/drive/MyDrive/CKIP/data")
ner = NER("/content/drive/MyDrive/CKIP/data")

# 讀取檔案並合併

In [ ]:
import json
# nuk停用頂端.json (用在斷詞前)
with open("/home/jupyter-selab/Chang/SortedCode/停用部分句/nuk停用頂端.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTonuk=[]
for word in stopwords:
  RemoveTonuk=set(word.get("stopwords", []))# 用set防止重複的東西
# print("toRemove:\n",toRemove)

# nuch停用頂端.json (用在斷詞前)
with open("/home/jupyter-selab/Chang/SortedCode/停用部分句/nchu停用頂端.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTonuchu=[]
for word in stopwords:
  RemoveTonuchu=set(word.get("stopwords", []))# 用set防止重複的東西

# 無主題式詞 (用在特徵選取後)
with open("/home/jupyter-selab/Chang/SortedCode/停用部分句/無主題式詞.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTheMeaningless=[]
for word in stopwords:
  RemoveTheMeaningless=set(word.get("stopwords", []))# 用set防止重複的東西

In [ ]:
# nkust
# 讀取 JSON 檔案
import json
import re
with open("/content/drive/MyDrive/collab-topic/斷詞前的資料/已有斷詞_程式不碰/nkust-topic.json", "r") as f:
    data = json.load(f)

valid_pos = {"Na", "Nb", "Nc", "Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VE", "VF", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}

nkustJsonFile=[]
# 對 JSON 檔案內的文字進行斷詞
counter=0
for item in data:
    # if counter==2: break
    # counter+=1
    # 取得文字內容
    text = ' '.join( item.get("title", None) )
    text = ' '.join( item.get("content", None) )
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)
    # CKIP 斷詞與詞性標註
    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])
    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]

    nkustJsonFile.append({
        "TitleAndDESC":featureWord,
        "Topic":["教育活動"],
    })
    # print("nkustJsonFile:\n",nkustJsonFile)
with open('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/nkust.json', 'w', encoding='utf-8') as f:
    json.dump(nkustJsonFile, f, ensure_ascii=False, indent=4)
print(len(nkustJsonFile)) # 269

In [ ]:
#nuk
import json

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞前的資料/已有斷詞_程式不碰/nuk.json", "r") as f:
    data = json.load(f)
nukJsonFile=[]
counter=0
# 對 JSON 檔案內的文字進行斷詞
for item in data:
    if counter==1: break
    counter+=1
    # 取得文字內容
    text = ' '.join(item.get("title", []))
    content=item.get("content", [])
    # for s in toRemove: # 這個也可以刪除停用詞
    #     while s in content:
    #         content.remove(s)
    
    # 和停用句json檔比對
    # 如果content內容有部分完全和停用句檔內一模一樣，那麼刪除content內容中有和停用句檔內一模一樣的內容
    content = [c for c in content if c not in RemoveTonuk]
    text = ' '.join(content)
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)
    # CKIP 斷詞與詞性標註
    print("text:\n"text)
    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])
    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]

    nukJsonFile.append({
        "TitleAndDESC":featureWord,
        "Topic":["教育活動"],
    })

# with open('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/nuk.json', 'w', encoding='utf-8') as f:
#     json.dump(nukJsonFile, f, ensure_ascii=False, indent=4)
# print(len(nukJsonFile)) # 215

In [ ]:
# 開放政府
import json
import glob

# 取得資料夾內所有的 JSON 檔案路徑
json_files = glob.glob('/content/drive/MyDrive/collab-topic/斷詞前的資料/開放政府_複製/*.json 的副本')
gov_json = []

# 逐一讀取 JSON 檔案
for json_file in json_files:
    # print(json_file)
    with open(json_file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        # counter=0
        for item in json_data:
          # if counter==1:  break
          # counter+=1
          text = ' '.join(item.get("Title", []))
          Description=item.get("Description", [])
          if Description != None and Description !=[]: # 防止加空(因為加None會error)
              text = ' '.join(Description)
          Topics = item.get("Topic", None)
          # 對Topics kry裡面放list or str 做處理
          if type(Topics) == str :
            Topics = [Topics]
          # print("text:\n",text)
          
          # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
          text = re.sub(r'[a-zA-Z]', ' ', text)
          # CKIP 斷詞與詞性標註
          doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
          ws_text=pipeline.get_ws(doc)
          pos_text=pipeline.get_pos(doc)
          # 只選特定詞性
          featureWord=[]
          for i,sent_p in enumerate(pos_text):
            for j,word_p in enumerate(sent_p):
              if word_p in valid_pos:
                featureWord.append(ws_text[i][j])
          # 刪除無意義詞(無主題式)-停用詞
          featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
          gov_json.append({
                    "TitleAndDESC":featureWord,
                    "Topic":Topics,
                })
          # print("featureWord:\n",featureWord)
# 將新的 JSON 資料存入檔案中。
with open('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/開放政府資料.json', 'w', encoding='utf-8') as f:
    json.dump(gov_json, f, ensure_ascii=False, indent=4)
print(len(gov_json)) # 1190

In [ ]:
# nchu
# nuch停用頂端.json (用在斷詞前)
with open("/content/drive/MyDrive/collab-topic/停用部分句/nchu停用頂端.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTonuchu=[]
for word in stopwords:
  RemoveTonuchu=set(word.get("stopwords", []))# 用set防止重複的東西

source='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/nchu-topic (1)'
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/nchu.json'
json_files = glob.glob(source+'/*.json')
newJsonFile = []
counter=0
# 逐一讀取 JSON 檔案
for json_file in json_files:
  # print(json_files) #1012
  # print(json_file) #1012
  # if counter==10:  break
  # counter=counter+1
  with open(json_file, 'r', encoding='utf-8') as f:
      json_data = json.load(f)
      # print('json_data:\n',json_data)
      # 取得文字內容
      text = ' '.join( json_data.get("title", []) )
      # print('title:\n',text)
      content_texts = json_data.get("content", [])
      # print('content_text_s(原):\n',content_texts)
      # 刪除nuhu頂端固定格式-停用詞
      for AstopWord in RemoveTonuchu:
        content_texts = [text.replace(AstopWord, ' ') for text in content_texts]

      for content_text in content_texts:
        text = text + ' ' + content_text
      # print('article text:\n',text)
      # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
      text = re.sub(r'[a-zA-Z]', ' ', text)
      # CKIP 斷詞與詞性標註
      doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
      ws_text=pipeline.get_ws(doc)
      pos_text=pipeline.get_pos(doc)
      # 只選特定詞性
      featureWord=[]
      for i,sent_p in enumerate(pos_text):
        for j,word_p in enumerate(sent_p):
          if word_p in valid_pos:
            featureWord.append(ws_text[i][j])
      # 刪除無意義詞(無主題式)-停用詞
      featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
      newJsonFile.append({
          "TitleAndDESC":featureWord,
          "Topic":["教育活動"],
      })
      # print(featureWord)

with open(destination, 'w', encoding='utf-8') as f:
  json.dump(newJsonFile, f, ensure_ascii=False, indent=4)
print(len(newJsonFile)) # 3050

In [ ]:
# content_texts = [c for c in content_texts if c not in RemoveTonuchu]
if 'a' in 'abc':
  print("1")
if 'a' in 'a b c':
  print("2")
if 'a b c' in 'a b c':
  print("3")



In [ ]:
# ntnu
source='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞前文件/ntnu-topic'
destination='/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/ntnu.json'
json_files = glob.glob(source+'/*.json')
newJsonFile = []
counter=0
# 逐一讀取 JSON 檔案
for json_file in json_files:
  # if counter==1:  break
  # counter=counter+1
  with open(json_file, 'r', encoding='utf-8') as f:
      json_data = json.load(f)
      # print('json_data:\n',json_data)
      # 取得文字內容
      text = ' '.join( json_data.get("title", []) )
      # print('title:\n',text)
      content_texts = json_data.get("content", [])
      # print('content_text_s:\n',content_texts)
      for content_text in content_texts:
        # print('content_text:\n',content_text)
        text = text + ' ' + content_text
      # print('article text:\n',text)
      # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
      text = re.sub(r'[a-zA-Z]', ' ', text)
      # CKIP 斷詞與詞性標註
      doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
      ws_text=pipeline.get_ws(doc)
      pos_text=pipeline.get_pos(doc)
      # 只選特定詞性
      featureWord=[]
      for i,sent_p in enumerate(pos_text):
        for j,word_p in enumerate(sent_p):
          if word_p in valid_pos:
            featureWord.append(ws_text[i][j])
      # 刪除無意義詞(無主題式)-停用詞
      featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]
      newJsonFile.append({
          "TitleAndDESC":featureWord,
          "Topic":["教育活動"],
      })
      # print(featureWord)

with open(destination, 'w', encoding='utf-8') as f:
  json.dump(newJsonFile, f, ensure_ascii=False, indent=4)
print(len(newJsonFile)) # 226

## 合併成一個json(nuk+nkust+ntnu+nuchu+開放政府資料-4950)

In [ ]:
import json
import glob

# 取得資料夾內所有的 JSON 檔案路徑
json_files = glob.glob('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/*.json')
merged_json = []

# 逐一讀取 JSON 檔案
for json_file in json_files:
    print(json_files)
    with open(json_file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        for item in json_data:
            TitleAndDESC = item.get("TitleAndDESC", None)
            Topic = item.get("Topic", None)
            if TitleAndDESC==None:
              print("TitleAndDESC_error")
            if Topic==None:
              print("Topic_error")
            
            merged_json.append({
                    "TitleAndDESC":TitleAndDESC,
                    "Topic":Topic,
                })
# +nkust看看，然後數有多少key or 元素
# 將新的 JSON 資料存入檔案中。
with open('/content/drive/MyDrive/collab-topic/重做斷詞/斷詞後/nuk+nkust+ntnu+nuchu+開放政府資料.json', 'w', encoding='utf-8') as f:
    json.dump(merged_json, f, ensure_ascii=False, indent=4)
print("總資料量：",len(merged_json)) # 4950

# lda 超參數

In [ ]:
def outputToTxt(content,d):# 變量內容, 目的地：用於將變量輸出到文字檔裡面觀察
  with open(d, "w") as f:
    f.write(str(content))
  return

In [ ]:
import os

# 獲取當前工作目錄路徑
cwd = os.getcwd()
print("當前工作目錄：", cwd)

In [ ]:
# 用來創(看)變數-texts、dictionary、corpus
import json
from gensim import corpora, models

# 讀取 JSON 檔案
with open('/home/jupyter-selab/Chang/SortedCode/nuk+nkust+ntnu+nuchu+開放政府資料.json', 'r') as f:
    data = json.load(f)

# 將詞彙列表合併為字串
docs = [" ".join(d["TitleAndDESC"]) for d in data]
# print("docs:\n",docs)
outputToTxt(docs,'docs.txt')

# 創建詞袋
texts = [[word for word in doc.split()] for doc in docs]
# print("texts:\n",texts)
dictionary = corpora.Dictionary(texts)
outputToTxt(texts,'texts.txt')
# print("dictionary:\n",dictionary)
outputToTxt(dictionary,'dictionary.txt')
corpus = [dictionary.doc2bow(text) for text in texts]
# print("corpus:\n",corpus)
outputToTxt(corpus,'corpus.txt')

# 創建LDA模型
# lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=10)

## 網格搜索 

In [ ]:
# 單進程版
from gensim.models.coherencemodel import CoherenceModel
import itertools
import numpy as np
from gensim.models import LdaModel
from gensim import corpora, models
# from multiprocessing import Pool, set_start_method

# 定义超参数空间
param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],# 6
        # 'alpha': [0.01, 0.1],#預設auto，會自動設置合適的
        # 'eta': [0.01, 0.1],
        'iterations': [600],
        'passes': [600]}

# 枚举所有超参数组合
param_list = list(itertools.product(*param_grid.values()))

# 定义模型评分函数
def evaluate_model(param):
    lda_model = LdaModel(corpus, id2word=dictionary, random_state=42, **dict(zip(param_grid.keys(), param)))
    coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
    coherence = coherence_model.get_coherence()
    perplex = lda_model.log_perplexity(corpus)# corpus_sparse改corpus
    return coherence, perplex

# 计算所有超参数组合的模型评分
scores = np.array([evaluate_model(param) for param in param_list])
# scores = np.array(pool.map(evaluate_model, param_list))
# 找到最优超参数组合
# best_param = param_list[scores.argmax()]
# print(f'Best param: {dict(zip(param_grid.keys(), best_param))}, score: {scores.max()}')
# 找到最优超参数组合
best_index = scores[:,0].argmax()# 从 scores 数组中选取所有行的第一列数据，然后返回最大值所在的索引。
best_param = param_list[best_index]
best_coherence = scores[best_index][0]
best_perplexity = scores[best_index][1] # 對於best_coherence而言的perplexity
print(f'Best param: {dict(zip(param_grid.keys(), best_param))}, coherence: {best_coherence}, perplexity: {best_perplexity}')

In [ ]:
import matplotlib.pyplot as plt

print("scores:\n",scores)
# 一致性折線圖
# 將一致性取出
coherence = [s[0] for s in scores]

# 設定 x 軸為評分的 index
x = range(2,11)#range(len(scores))

# 繪製折線圖(越大越好)
plt.plot(x, coherence, '-o')

# 設定圖片標題和軸標籤
plt.title('Coherence Scores')
plt.xlabel('Index')
plt.ylabel('Coherence')

# 顯示圖片
plt.show()

# 困惑度折線圖(越小越好)
# 將一致性取出
perplexity = [s[1] for s in scores]

# 設定 x 軸為評分的 index
x = range(2,11)#range(len(scores))

# 繪製折線圖
plt.plot(x, perplexity, '-o')

# 設定圖片標題和軸標籤
plt.title('perplexity Scores')
plt.xlabel('Index')
plt.ylabel('perplexity')

# 顯示圖片
plt.show()

param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],# 9
        # 'alpha': [0.01, 0.1],#預設auto，會自動設置合適的
        # 'eta': [0.01, 0.1],
        'iterations': [60],
        'passes': [30]}
scores:
 [[ 0.44532239 -8.1103544 ]#大部分都是2有最低的困惑度
 [ 0.45566517 -8.05484744]
 [ 0.47150508 -8.01658246]
 [ 0.48896125 -7.98693406]#試試(次高一致性，但困惑度約第3高)
 [ 0.48499323 -7.99589956]
 [ 0.45839687 -8.00056021]
 [ 0.47887587 -7.98635326]
 [ 0.48975793 -7.95369813]#試試(最高一致性，但困惑度也最高)
 [ 0.45930634 -7.95765243]]
 

---
param_grid = {'num_topics': [2,3,4,5,6,7,8,9,10],# 9
        # 'alpha': [0.01, 0.1],#預設auto，會自動設置合適的
        # 'eta': [0.01, 0.1],
        'iterations': [200],
        'passes': [200]}
scores:
 [[ 0.46937707 -8.10298631]#大部分都是2有最低的困惑度
 [ 0.46425728 -8.03825614]
 [ 0.46616865 -7.99523772]
 [ 0.46155753 -7.98522191]
 [ 0.44870283 -8.00851874]
 [ 0.46704819 -8.01126265]
 [ 0.46663949 -8.01368787]
 [ 0.50746007 -7.99944476]#一致性最高，困惑度排倒數第4
 [ 0.50224944 -7.99412277]]#一致性次高，困惑度排倒數第2



## 訓練模型

In [ ]:
num_topics=9
iterations=200
passes=200
# 训练主题模型
lda_model = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, iterations=iterations,passes=passes, random_state=42)


## 可視化

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

# 創建可視化資料
vis_data = gensimvis.prepare(topic_model=lda_model, corpus=corpus, dictionary=dictionary)

# 将数据保存到JSON文件

pyLDAvis.save_json(vis_data, '/home/jupyter-selab/Chang/SortedCode/experimentResult/0227(num_topics=9,iterations=200,passes=200).json')
# ChangSortedCodeexperimentResult
# 顯示可視化
pyLDAvis.display(vis_data)

# 保存為HTML文件
pyLDAvis.save_html(vis_data, '/home/jupyter-selab/Chang/SortedCode/experimentResult/0227(num_topics=9,iterations=200,passes=200).html')

# 預測新文本

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})
valid_pos = {"Na", "Nb", "Nc", "Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VE", "VF", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}

# pipeline = CkipPipeline()  # create a CKIP pipeline object
def preprocess_text(text):
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)

    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
    # 只選特定詞性
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        if word_p in valid_pos:
          featureWord.append(ws_text[i][j])
    # 刪除無意義詞(無主題式)-停用詞
    featureWord = [c for c in featureWord if c not in RemoveTheMeaningless]

    return featureWord

new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動'

# 預處理新文本，建立詞彙表
new_doc = preprocess_text(new_text)
print(new_text)
print(new_doc)

In [ ]:
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import gensim.downloader as api

# 加載訓練好的主題模型
# model_name = 'lda_model'
# model = api.load(model_name)
model = lda_model
# 創建新文本
new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動'
print("文本:\n",new_text)

# 預處理新文本，建立詞彙表
new_doc = preprocess_text(new_text)
print("特徵選取後:\n",new_doc)
# dictionary = Dictionary.load(train_dictionary)
# dictionary = train_dictionary

# 將新文本轉換為詞袋表示
new_doc_bow = dictionary.doc2bow(new_doc)

# 預測新文本的主題
doc_topics = model.get_document_topics(new_doc_bow)

# 輸出主題預測結果
print(doc_topics)

In [ ]:
def predictArticle(new_text):
    # 加載訓練好的主題模型
    # model_name = 'lda_model'
    # model = api.load(model_name)
    model = lda_model
    # 創建新文本
#     new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動'
#     print("文本:\n",new_text)

    # 預處理新文本，建立詞彙表
    new_doc = preprocess_text(new_text)
#     print("特徵選取後:\n",new_doc)
    # dictionary = Dictionary.load(train_dictionary)
    # dictionary = train_dictionary

    # 將新文本轉換為詞袋表示
    new_doc_bow = dictionary.doc2bow(new_doc)

    # 預測新文本的主題
    doc_topics = model.get_document_topics(new_doc_bow)
    
    return doc_topics

In [ ]:
# from nknu 高教深耕活動一覽表(成果展)
# http://osas.nknu.edu.tw/HESP/1160
nkust1='''
原資中心-kinizala 共享成果展
發消息~~
✨原資中心成果展來啦！✨

趕快找你的朋友參加一年一度原資成果展。這次成果展有 *靜態展覽、晚會* ，歡迎大家跟我們一同參與，一起欣賞大家獨一無二的作品。
最重要的12/23(五)晚上6:00-9:00有我們最愛的晚會活動，讓我們一邊欣賞樂舞唱跳、一邊享受著美食佳餚。
'''
predictArticle(nkust1)

In [ ]:
# from nknu 校園活動(志工服務相關)
# https://sso.nknu.edu.tw/Services/CampusActivities/ActUser.aspx
nknu1='''
福愛課後照顧班
於4/15、4/22、4/29、5/13、5/20、6/03、6/04 至苓洲國小及光榮國小進行課輔活動
'''
predictArticle(nknu1)

# 导入相关包

In [ ]:
!pip install opencc
!pip install pyLDAvis
!pip install gensim

In [ ]:
# from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt
import pyLDAvis.gensim_models as gensimvis #可視化
import pyLDAvis #可視化
import json
import opencc # 簡轉繁用
import jieba
import re
import os